# Customer segmentation with K-means++

A small clothing retailer had customer records but no segmentation — every customer got the same range at the same price. This notebook clusters the customer base and identifies which segment to design and price the next collection around.

**Data availability:** assumes `data/customers.csv` (2,000 rows, 7 attributes: gender, marital status, age, education, occupation, income, settlement size), not redistributed here. Place the file under `data/` to run this notebook end-to-end.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("data/customers.csv")
customers.info()

## Exploratory data analysis

Univariate splits on the raw attributes, reproducing the verified figures reported in the case study.

In [ ]:
verified_splits = {
    "gender": {"male": 54.3, "female": 45.7},
    "marital_status": {"single": 50.3, "not_single": 49.7},
    "education": {"high_school": 69.3, "undergraduate": 14.6, "graduate": 1.8},
    "occupation": {"skilled_or_official": 55.7, "unemployed_or_unskilled": 31.7, "managers": 12.7},
}
# pct = customers["gender"].value_counts(normalize=True) * 100
pd.DataFrame(verified_splits).T

The education split (69.3 + 14.6 + 1.8 = 85.7%) does not sum to 100 from the supplied figures; the remaining 14.3% is reported as "other / unspecified" rather than assigned to an invented category.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_cols = ["age", "income"]
categorical_cols = ["gender", "marital_status", "education", "occupation", "settlement_size"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])
# X = preprocess.fit_transform(customers)

## Choosing k

Cluster count was chosen with an elbow analysis on inertia across k = 2..8, cross-checked against agglomerative clustering — both converged on **three** segments. The per-k inertia values from that run are not recorded in this repository, so the elbow curve is described here rather than re-plotted from invented numbers; the code below reproduces the method.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

# inertias = [KMeans(n_clusters=k, init="k-means++", random_state=0).fit(X).inertia_
#             for k in range(2, 9)]

kmeans = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=0)
agglomerative = AgglomerativeClustering(n_clusters=3)

# kmeans_labels = kmeans.fit_predict(X)
# agglo_labels = agglomerative.fit_predict(X)
# agreement = adjusted_rand_score(kmeans_labels, agglo_labels)
# print(f"Adjusted Rand index between the two methods: {agreement:.3f}")

## Segment profiles

The three resulting segments, profiled against the verified age/income facts reported in the case study:

In [ ]:
segments = pd.DataFrame([
    {"segment": "Lower", "age": 31, "income": 70_000, "profile": "female-leaning, non-single, high school, unemployed/unskilled, small city"},
    {"segment": "Mid (target)", "age": 38, "income": 130_000, "profile": "male-leaning, single, high school, skilled/official, small-mid city"},
    {"segment": "Upper", "age": 45, "income": 250_000, "profile": "highest age/income, smallest group"},
])
segments

## Recommendation

Design and price the next collection around the **Mid segment** — the group with the disposable income (~$130k) and lifestyle profile (single, skilled, city-based) most likely to convert. Use the Lower profile to justify a secondary value line, and the Upper profile (smallest group, highest income) to justify a limited premium line rather than a core range.

## What I'd do next — explored

Two follow-ups: keep the two clustering methods as a standing cross-check, and keep every segment profile traceable to the marketing action it justifies. Both are worked through below.

### 1. Cross-check: quantitative agreement between methods

K-means++ and agglomerative clustering were reported as agreeing on three segments — that claim is currently qualitative. The Adjusted Rand Index (ARI) turns it into a single number between -1 and 1 (1 = identical partitions, 0 = agreement no better than chance). It should be re-run and reported alongside the segment profiles every time this notebook runs against the real data — no ARI value is printed here because it hasn't been recomputed in this session.

In [ ]:
from sklearn.metrics import adjusted_rand_score

# kmeans_labels = kmeans.fit_predict(X)
# agglo_labels = agglomerative.fit_predict(X)
# ari = adjusted_rand_score(kmeans_labels, agglo_labels)
# print(f"Adjusted Rand index: {ari:.3f}")
# A value above ~0.5 supports treating the 3-segment structure as robust
# to the choice of clustering algorithm; this threshold is a common
# rule of thumb, not a property proven for this dataset specifically.

### 2. Segment profile → recommendation mapping

Every segment profile should trace to one marketing action. This table makes that link explicit, built only from the verified age/income/attribute facts reported above.

In [ ]:
profile_to_action = pd.DataFrame([
    {"segment": "Lower", "dominant_attributes": "~31yo, ~$70k, female-leaning, non-single, high school, unemployed/unskilled, small city", "recommended_action": "Secondary value-priced line."},
    {"segment": "Mid (target)", "dominant_attributes": "~38yo, ~$130k, male-leaning, single, high school, skilled/official, small-mid city", "recommended_action": "Design and price the next collection around this segment."},
    {"segment": "Upper", "dominant_attributes": "~45yo, ~$250k, smallest group", "recommended_action": "Limited premium line only."},
])
profile_to_action